# LeetCode 935 — Knight Dialer bằng Python

Link đề: https://leetcode.com/problems/knight-dialer/ · Độ khó: **Medium** · Chủ đề: Dynamic Programming

Skeleton LeetCode:

```python
class Solution:
    def knightDialer(self, n: int) -> int:
        ...
```

Ghi chú: đáp án duy nhất (một số nguyên, lấy **modulo `10^9 + 7`**). Điểm cốt lõi: số số điện thoại độ dài `n` kết thúc ở chữ số `d` bằng **tổng** số số độ dài `n − 1` kết thúc ở các chữ số mà quân mã nhảy được tới `d`: quy hoạch động theo **chữ số cuối**, `O(10 n)`. Có thể rút gọn nhờ **đối xứng** của bàn phím (còn 4 trạng thái) hoặc dùng **luỹ thừa ma trận** `10 × 10` cho `n` rất lớn. Oracle: **liệt kê mọi dãy nhảy** của quân mã trên bàn phím (kiểm tra nước đi bằng hình học `(±1, ±2)`, `(±2, ±1)`).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: quân mã đặt ở một ô số bất kỳ của bàn phím điện thoại, thực hiện `n − 1` bước nhảy **hợp lệ** (chỉ đáp xuống ô số); mỗi dãy ô đã đi qua ghi thành một số điện thoại độ dài `n`; đếm số dãy phân biệt.
2. Nắm công cụ Python: danh sách kề, quy hoạch động theo trạng thái "chữ số hiện tại", `% MOD`, nhân/luỹ thừa ma trận modulo, tính đối xứng.
3. Đi từ **liệt kê mọi dãy (brute force `O(2.x^n)`)** → **DP theo chữ số cuối (`O(10 n)`)** → **DP rút gọn nhờ đối xứng (4 trạng thái)** → **luỹ thừa ma trận (`O(10³ log n)`)**.
4. Hiểu mấu chốt: bàn phím có **đồ thị nước đi cố định** (`10` đỉnh, `20` cạnh có hướng); số dãy độ dài `n` là số **đường đi** độ dài `n − 1` trong đồ thị đó; truy hồi tuyến tính ⇒ ma trận kề luỹ thừa.
5. Tránh lỗi kinh điển: kề sai ô (nhất là `4` và `6` có `3` láng giềng, `5` không có), nhầm số bước (`n − 1` bước chứ không phải `n`), quên `% MOD`, cập nhật mảng DP **tại chỗ** (làm hỏng giá trị còn cần), và tính chất đối xứng áp dụng sai.

## 2. Đề bài & ví dụ

Quân mã cờ vua nhảy theo hình chữ `L`: đi `2` ô theo một phương rồi `1` ô theo phương vuông góc. Bàn phím điện thoại:

```
1 2 3
4 5 6
7 8 9
  0
```

Quân mã **chỉ được đứng trên ô số** (10 ô). Cho số nguyên `n`: đếm số **số điện thoại phân biệt** độ dài `n` có thể quay: đặt quân mã ở **ô số bất kỳ** rồi thực hiện `n − 1` bước nhảy hợp lệ; ghi lại các chữ số theo thứ tự đã đứng.

Đáp án có thể rất lớn: trả về **modulo `10^9 + 7`**.

**Ví dụ 1:** `n = 1` → `10`.

**Ví dụ 2:** `n = 2` → `20` (các số `04, 06, 16, 18, 27, 29, 34, 38, 40, 43, 49, 60, 61, 67, 72, 76, 81, 83, 92, 94`).

**Ví dụ 3:** `n = 3131` → `136006598`.

**Ràng buộc:** `1 <= n <= 5000`.

In [1]:
import itertools
import random
import time
from typing import List

MOD = 10 ** 9 + 7
examples = [(1, 10), (2, 20), (3131, 136006598)]
for n, want in examples:
    print(f"n = {n} -> {want}")

n = 1 -> 10
n = 2 -> 20
n = 3131 -> 136006598


## 3. Kiến thức nền

### 3.1 Toạ độ bàn phím và nước nhảy của quân mã

Đặt `1..9` ở lưới `3 × 3` (hàng `0..2`, cột `0..2`) và `0` ở hàng `3`, cột `1`. Quân mã từ `(r, c)` tới `(r + dr, c + dc)` với `(dr, dc) ∈ {(±1, ±2), (±2, ±1)}`, miễn là ô đích là ô số.

In [2]:
POS = {1: (0, 0), 2: (0, 1), 3: (0, 2), 4: (1, 0), 5: (1, 1), 6: (1, 2), 7: (2, 0), 8: (2, 1), 9: (2, 2), 0: (3, 1)}
CELL = {p: d for d, p in POS.items()}
KNIGHT = [(1, 2), (1, -2), (-1, 2), (-1, -2), (2, 1), (2, -1), (-2, 1), (-2, -1)]


def moves_from_geometry(d):
    r, c = POS[d]
    return sorted(CELL[(r + dr, c + dc)] for dr, dc in KNIGHT if (r + dr, c + dc) in CELL)


ADJ = {d: moves_from_geometry(d) for d in range(10)}
print(ADJ)
print("tổng số cạnh:", sum(len(v) for v in ADJ.values()), "| ô 5 có", len(ADJ[5]), "nước đi")

{0: [4, 6], 1: [6, 8], 2: [7, 9], 3: [4, 8], 4: [0, 3, 9], 5: [], 6: [0, 1, 7], 7: [2, 6], 8: [1, 3], 9: [2, 4]}
tổng số cạnh: 20 | ô 5 có 0 nước đi


### 3.2 Đồ thị đối xứng

Đồ thị nước đi là **vô hướng** (`a → b` ⇔ `b → a`) và có đối xứng trái–phải: `1 ↔ 3`, `4 ↔ 6`, `7 ↔ 9` (phản chiếu qua cột giữa) và các phép đối xứng lên–xuống chỉ giữ nguyên trong tập `{1,3,7,9}` (nhóm "góc"). Ô `5` cô lập (không có nước đi). Ô `0` đối xứng với chính nó.

### 3.3 DP theo chữ số cuối

`dp[k][d]` = số dãy độ dài `k` kết thúc ở `d`. `dp[1][d] = 1` với mọi `d`; `dp[k][d] = Σ_{s: s → d} dp[k−1][s]`. Đáp án `Σ_d dp[n][d]` (modulo). Chỉ cần giữ hàng `dp[k−1]` để tính `dp[k]` (cuộn).

### 3.4 Dạng ma trận

Gọi `A` là ma trận kề `10 × 10` (`A[s][d] = 1` nếu quân mã nhảy được `s → d`). Vector `v_k = (dp[k][0..9])` thoả `v_k = v_{k−1} · A`. Do đó `v_n = v_1 · A^{n−1}` và đáp án là tổng các phần tử của `v_1 · A^{n−1}` (với `v_1` toàn `1`). Luỹ thừa nhanh `A^{n−1}` tốn `O(10³ log n)`.

In [3]:
A = [[1 if d in ADJ[s] else 0 for d in range(10)] for s in range(10)]
print("A đối xứng:", all(A[i][j] == A[j][i] for i in range(10) for j in range(10)))
v = [1] * 10
for k in range(2, 6):
    v = [sum(v[s] * A[s][d] for s in range(10)) for d in range(10)]
    print(f"k = {k}: v = {v}, tổng = {sum(v)}")

A đối xứng: True
k = 2: v = [2, 2, 2, 2, 3, 0, 3, 2, 2, 2], tổng = 20
k = 3: v = [6, 5, 4, 5, 6, 0, 6, 5, 4, 5], tổng = 46
k = 4: v = [12, 10, 10, 10, 16, 0, 16, 10, 10, 10], tổng = 104
k = 5: v = [32, 26, 20, 26, 32, 0, 32, 26, 20, 26], tổng = 240


## 4. Ý tưởng nền tảng / trực giác

`n = 2`: mỗi chữ số xuất phát có bao nhiêu nước nhảy? `0 → {4, 6}`, `1 → {6, 8}`, `2 → {7, 9}`, `3 → {4, 8}`, `4 → {0, 3, 9}`, `5 → {}`, `6 → {0, 1, 7}`, `7 → {2, 6}`, `8 → {1, 3}`, `9 → {2, 4}`. Tổng `2 + 2 + 2 + 2 + 3 + 0 + 3 + 2 + 2 + 2 = 20` ✓ (khớp ví dụ 2).

`n = 3`: với mỗi số hai chữ số `ab`, thêm một nước nhảy từ `b`: tổng `Σ_b (số số 2 chữ số kết thúc ở b) · deg(b)`. Số số kết thúc ở mỗi chữ số (k=2): `[2,2,2,2,3,0,3,2,2,2]`; nhân với bậc: `2·2 + 2·2 + 2·2 + 2·2 + 3·3 + 0 + 3·3 + 2·2 + 2·2 + 2·2 = 46` ✓.

In [4]:
c2 = [sum(1 for s in range(10) if d in ADJ[s]) for d in range(10)]           # số số 2 chữ số kết thúc ở d
print("kết thúc ở d (k=2):", c2, "| tổng", sum(c2))
print("k=3:", sum(c2[b] * len(ADJ[b]) for b in range(10)))

kết thúc ở d (k=2): [2, 2, 2, 2, 3, 0, 3, 2, 2, 2] | tổng 20
k=3: 46


## 5. Cách 1 — Brute Force: liệt kê mọi dãy nhảy

**Ý tưởng:** DFS từ mỗi chữ số xuất phát, đệ quy `n − 1` bước theo danh sách kề, đếm số dãy hoàn chỉnh.

In [5]:
def knight_brute(n):
    def dfs(d, left):
        if left == 0:
            return 1
        return sum(dfs(nxt, left - 1) for nxt in ADJ[d])

    return sum(dfs(d, n - 1) for d in range(10)) % MOD


for n, want in examples[:2]:
    assert knight_brute(n) == want
print([knight_brute(n) for n in range(1, 9)])

[10, 20, 46, 104, 240, 544, 1256, 2848]


In [6]:
# Code nộp LeetCode cho Cách 1 (O(2.x^n): chỉ đúng với n nhỏ; TLE với n lớn)
class SolutionBrute:
    def knightDialer(self, n: int) -> int:
        adj = {0: [4, 6], 1: [6, 8], 2: [7, 9], 3: [4, 8], 4: [0, 3, 9], 5: [], 6: [0, 1, 7], 7: [2, 6], 8: [1, 3], 9: [2, 4]}

        def dfs(d, left):
            if left == 0:
                return 1
            return sum(dfs(nxt, left - 1) for nxt in adj[d])

        return sum(dfs(d, n - 1) for d in range(10)) % (10 ** 9 + 7)

**Độ phức tạp:** số dãy tăng như `~2.2^n`; `n = 30` đã không thực tế. Bộ nhớ `O(n)` (ngăn xếp đệ quy).

**Nút thắt:** cùng một cặp `(chữ số hiện tại, số bước còn lại)` được tính lặp lại rất nhiều lần.

## 6. Ý tưởng thuật toán / bước đột phá

Số dãy tiếp theo chỉ phụ thuộc `(chữ số hiện tại, số bước còn lại)` — `10 · n` trạng thái. DP cuộn theo `n`: `dp_new[d] = Σ_{s → d} dp_old[s]`. Với `n` cực lớn: luỹ thừa ma trận.

## 7. Các cách tiếp theo

Ngoài Cách 1 (mục 5), ta có thêm 3 cách: **Cách 2** DP theo chữ số cuối, **Cách 3** DP rút gọn nhờ đối xứng, **Cách 4** luỹ thừa ma trận.

### Cách 2 — DP theo chữ số cuối

**Giải thích từng đoạn:**

1. `dp = [1] * 10` (độ dài 1).
2. Lặp `n − 1` lần: `new[d] = Σ_{s: d ∈ nxt[s]} dp[s]`; cách viết "đẩy": với mỗi `s`, với mỗi `d` trong `nxt[s]` cộng `dp[s]` vào `new[d]`.
3. Trả `Σ dp % MOD`.

In [7]:
NXT = {0: [4, 6], 1: [6, 8], 2: [7, 9], 3: [4, 8], 4: [0, 3, 9], 5: [], 6: [0, 1, 7], 7: [2, 6], 8: [1, 3], 9: [2, 4]}
assert NXT == ADJ                                        # danh sách kề viết tay khớp tính từ hình học


def knight_dp(n):
    dp = [1] * 10                                        # độ dài 1: mỗi chữ số một cách
    for _ in range(n - 1):
        new = [0] * 10                                   # mảng mới: không cập nhật tại chỗ
        for s in range(10):
            for d in NXT[s]:
                new[d] = (new[d] + dp[s]) % MOD
        dp = new
    return sum(dp) % MOD


for n, want in examples:
    assert knight_dp(n) == want
print([knight_dp(n) for n, _ in examples], [knight_dp(n) for n in range(1, 9)])

[10, 20, 136006598] [10, 20, 46, 104, 240, 544, 1256, 2848]


In [8]:
# Code nộp LeetCode cho Cách 2
class SolutionDP:
    def knightDialer(self, n: int) -> int:
        MOD = 10 ** 9 + 7
        nxt = [[4, 6], [6, 8], [7, 9], [4, 8], [0, 3, 9], [], [0, 1, 7], [2, 6], [1, 3], [2, 4]]
        dp = [1] * 10
        for _ in range(n - 1):
            new = [0] * 10
            for s in range(10):
                for d in nxt[s]:
                    new[d] = (new[d] + dp[s]) % MOD
            dp = new
        return sum(dp) % MOD

**Độ phức tạp:** `O(20 n)` thời gian (`20` cạnh), `O(1)` bộ nhớ (hai mảng `10` phần tử).

### Cách 3 — DP rút gọn nhờ đối xứng

**Ý tưởng:** các chữ số cùng "lớp" có cùng số dãy ở mọi độ dài: `A = {1, 3, 7, 9}` (góc), `B = {2, 8}`, `C = {4, 6}`, `D = {0}` (và `5` cô lập, chỉ đóng góp ở `n = 1`). Gọi `a, b, c, d` là số dãy kết thúc ở **một** phần tử của mỗi lớp. Từ nước đi: `1 ← {6, 8}`, `2 ← {7, 9}`, `4 ← {0, 3, 9}`, `0 ← {4, 6}`. Do đó
`a' = c + b`, `b' = 2a`, `c' = d + 2a`, `d' = 2c` (và tổng `= 4a + 2b + 2c + d` cho `n ≥ 2`, `+1` cho ô `5` khi `n = 1`).

In [9]:
def knight_symmetric(n):
    if n == 1:
        return 10
    a = b = c = d = 1                                    # a: ô góc {1,3,7,9}; b: {2,8}; c: {4,6}; d: {0} (mỗi biến: số dãy kết thúc ở MỘT ô của lớp)
    for _ in range(n - 1):
        a, b, c, d = (b + c) % MOD, 2 * a % MOD, (d + 2 * a) % MOD, 2 * c % MOD
    return (4 * a + 2 * b + 2 * c + d) % MOD


for n, want in examples:
    assert knight_symmetric(n) == want
print([knight_symmetric(n) for n, _ in examples], [knight_symmetric(n) for n in range(1, 9)])

[10, 20, 136006598] [10, 20, 46, 104, 240, 544, 1256, 2848]


In [10]:
# Code nộp LeetCode cho Cách 3
class SolutionSymmetric:
    def knightDialer(self, n: int) -> int:
        MOD = 10 ** 9 + 7
        if n == 1:
            return 10
        a = b = c = d = 1
        for _ in range(n - 1):
            a, b, c, d = (b + c) % MOD, 2 * a % MOD, (d + 2 * a) % MOD, 2 * c % MOD
        return (4 * a + 2 * b + 2 * c + d) % MOD

**Độ phức tạp:** `O(n)` thời gian với hằng số rất nhỏ (4 biến), `O(1)` bộ nhớ.

### Cách 4 — Luỹ thừa ma trận

**Ý tưởng:** `A^{n−1}` bằng bình phương liên tiếp (modulo); đáp án `= Σ_{i, j} (A^{n−1})[i][j]` (vì `v_1` toàn `1`). Hữu ích khi `n` lên tới `10^18` (bài tổng quát hoá).

In [11]:
def mat_mul(X, Y):
    n, m, p = len(X), len(Y), len(Y[0])
    return [[sum(X[i][k] * Y[k][j] for k in range(m)) % MOD for j in range(p)] for i in range(n)]


def mat_pow(M, e):
    size = len(M)
    R = [[1 if i == j else 0 for j in range(size)] for i in range(size)]
    while e:
        if e & 1:
            R = mat_mul(R, M)
        M = mat_mul(M, M)
        e >>= 1
    return R


def knight_matrix(n):
    P = mat_pow(A, n - 1)                                # A: ma trận kề 10 x 10
    return sum(sum(row) for row in P) % MOD              # v_1 = (1, ..., 1) => tổng mọi phần tử của A^(n-1)


for n, want in examples:
    assert knight_matrix(n) == want
print([knight_matrix(n) for n, _ in examples], [knight_matrix(n) for n in range(1, 9)])

[10, 20, 136006598] [10, 20, 46, 104, 240, 544, 1256, 2848]


In [12]:
# Code nộp LeetCode cho Cách 4
class SolutionMatrix:
    def knightDialer(self, n: int) -> int:
        MOD = 10 ** 9 + 7
        nxt = [[4, 6], [6, 8], [7, 9], [4, 8], [0, 3, 9], [], [0, 1, 7], [2, 6], [1, 3], [2, 4]]
        A = [[0] * 10 for _ in range(10)]
        for s in range(10):
            for d in nxt[s]:
                A[s][d] = 1

        def mul(X, Y):
            return [[sum(X[i][k] * Y[k][j] for k in range(10)) % MOD for j in range(10)] for i in range(10)]

        R = [[int(i == j) for j in range(10)] for i in range(10)]
        e = n - 1
        while e:
            if e & 1:
                R = mul(R, A)
            A = mul(A, A)
            e >>= 1
        return sum(map(sum, R)) % MOD

**Độ phức tạp:** `O(10³ log n)` thời gian (`~10³ · 13` với `n ≤ 5000`), `O(100)` bộ nhớ. Vượt trội khi `n` rất lớn.

## 8. TRACE từng bước

**Trace Cách 2** với `n = 4`:

| `k` | `dp` (kết thúc ở `0..9`) | tổng |
|---|---|---|
| 1 | `[1,1,1,1,1,1,1,1,1,1]` | 10 |
| 2 | `[2,2,2,2,3,0,3,2,2,2]` | 20 |
| 3 | `[6,5,4,5,6,0,6,5,4,5]` | 46 |
| 4 | `[12,10,10,10,16,0,16,10,10,10]` | 104 |

Hàm trace in `dp` thật và kiểm tra đối chiếu Cách 3 (đối xứng; để ý các ô cùng lớp `{1,3,7,9}`, `{2,8}`, `{4,6}` có cùng giá trị):

In [13]:
def trace_dp(n):
    dp = [1] * 10
    print(f"k = 1: dp = {dp}, tổng = {sum(dp)}")
    for k in range(2, n + 1):
        new = [0] * 10
        for s in range(10):
            for d in NXT[s]:
                new[d] += dp[s]
        dp = new
        print(f"k = {k}: dp = {dp}, tổng = {sum(dp)}")
    return sum(dp)


assert trace_dp(4) == 104 == knight_symmetric(4) == knight_matrix(4)

k = 1: dp = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1], tổng = 10
k = 2: dp = [2, 2, 2, 2, 3, 0, 3, 2, 2, 2], tổng = 20
k = 3: dp = [6, 5, 4, 5, 6, 0, 6, 5, 4, 5], tổng = 46
k = 4: dp = [12, 10, 10, 10, 16, 0, 16, 10, 10, 10], tổng = 104


**Trace Cách 3** (đối xứng): `(a, b, c, d)` bắt đầu `(1, 1, 1, 1)`. Bước 1: `(b + c, 2a, d + 2a, 2c) = (2, 2, 3, 2)` ⇒ tổng `4·2 + 2·2 + 2·3 + 2 = 20` ✓. Bước 2: `(2 + 3, 2·2, 2 + 4, 2·3) = (5, 4, 6, 6)` ⇒ `20 + 8 + 12 + 6 = 46` ✓.

## 9. Code nộp LeetCode

Chọn **Cách 2** (DP theo chữ số cuối; rõ ràng, `O(20 n)`). Giữ nguyên `class Solution`, `knightDialer(self, n)`:

In [14]:
class Solution:
    def knightDialer(self, n: int) -> int:
        MOD = 10 ** 9 + 7
        # nxt[s]: các ô mà quân mã nhảy tới được từ ô s
        nxt = [[4, 6], [6, 8], [7, 9], [4, 8], [0, 3, 9], [], [0, 1, 7], [2, 6], [1, 3], [2, 4]]
        dp = [1] * 10                                      # dp[d]: số dãy độ dài hiện tại kết thúc ở d
        for _ in range(n - 1):                             # n - 1 bước nhảy
            new = [0] * 10                                 # mảng mới: tránh cập nhật tại chỗ
            for s in range(10):
                for d in nxt[s]:
                    new[d] = (new[d] + dp[s]) % MOD
            dp = new
        return sum(dp) % MOD


print([Solution().knightDialer(n) for n, _ in examples])      # [10, 20, 136006598]
assert all(Solution().knightDialer(n) == want for n, want in examples)

[10, 20, 136006598]


## 10. Vì sao đúng

**(a) Đồ thị nước đi.** Một số điện thoại độ dài `n` ứng với dãy ô `d_1, d_2, …, d_n` mà mỗi cặp liên tiếp `(d_i, d_{i+1})` là một nước nhảy hợp lệ (cạnh trong đồ thị). Hai dãy ô khác nhau cho hai số khác nhau (số chính là dãy chữ số). Vậy đáp án = số **đường đi** (có thể lặp đỉnh) gồm `n` đỉnh, tức `n − 1` cạnh.

**(b) DP.** Gọi `dp_k[d]` = số đường đi `k` đỉnh kết thúc ở `d`. Một đường đi `k` đỉnh kết thúc ở `d` là một đường đi `k − 1` đỉnh kết thúc ở một đỉnh `s` có cạnh `s → d`, cộng cạnh cuối: `dp_k[d] = Σ_{s → d} dp_{k−1}[s]`, `dp_1[d] = 1`. Tổng `Σ_d dp_n[d]` đếm mọi đường đi. Dùng mảng **mới** mỗi bước vì `dp_{k−1}[s]` còn cần cho nhiều `d`.

**(c) Đối xứng (Cách 3).** Gọi `a_k, b_k, c_k, d_k` là `dp_k` tại một ô của lớp góc `{1,3,7,9}`, lớp `{2,8}`, lớp `{4,6}` và ô `0`. Truy hồi trên đồ thị: `1 ← {6, 8}`, `3 ← {4, 8}`, `7 ← {2, 6}`, `9 ← {2, 4}` (mỗi ô góc nhận từ một ô lớp `{4,6}` và một ô lớp `{2,8}`); `2 ← {7, 9}`, `8 ← {1, 3}` (hai ô góc); `4 ← {0, 3, 9}`, `6 ← {0, 1, 7}` (ô `0` và hai ô góc); `0 ← {4, 6}` (hai ô lớp `{4,6}`). Nếu ở bước `k − 1` mọi ô trong cùng lớp có cùng giá trị thì các công thức trên cho cùng giá trị ở bước `k` cho mọi ô trong lớp: `a' = b + c`, `b' = 2a`, `c' = d + 2a`, `d' = 2c`. Ở `k = 1` mọi ô bằng `1` nên giả thiết đúng; quy nạp theo `k`. (Ô `5` có `dp = 1` ở `k = 1` rồi `0`, nên chỉ cộng `1` khi `n = 1`.)

**(d) Ma trận (Cách 4).** `dp_k = dp_{k−1} · A` với `A[s][d] = [s → d]`; vậy `dp_n = dp_1 · A^{n−1}` và tổng các phần tử của `dp_n` là tổng mọi phần tử của `A^{n−1}` (vì `dp_1` toàn `1`). Luỹ thừa nhanh dùng tính kết hợp của nhân ma trận (modulo bảo toàn phép `+`, `×`).

**(e) Modulo.** Mọi phép cộng/nhân lấy `% MOD` giữ đúng kết quả modulo; Python không tràn nhưng ở ngôn ngữ khác phải dùng `long`.

Ta kiểm chứng bằng máy: mọi cách khớp oracle **liệt kê dãy nhảy theo hình học** với `n = 1..11` (mục 12D), khớp nhau tới `n = 5000`, và ví dụ `n = 3131` cho `136006598`.

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Liệt kê mọi dãy | `O(2.x^n)` | `O(n)` | oracle, `n ≤ 11` |
| 2. DP theo chữ số cuối | `O(20 n)` | `O(1)` | **đáp án chính** |
| 3. DP đối xứng | `O(n)` | `O(1)` | hằng số nhỏ nhất |
| 4. Luỹ thừa ma trận | `O(10³ log n)` | `O(100)` | `n` rất lớn |

Đo thực tế (`n = 5000`; luỹ thừa ma trận cho `n = 10^18`):

In [15]:
def bench(fn, arg, repeat=3):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(arg)
        best = min(best, time.perf_counter() - t0)
    return best


for name, fn in [("dp", knight_dp), ("symmetric", knight_symmetric), ("matrix", knight_matrix)]:
    print(f"n = 5000 {name:10s} {bench(fn, 5000) * 1e3:8.3f} ms")
print(f"matrix n = 10^18: {bench(knight_matrix, 10 ** 18) * 1e3:8.3f} ms ->", knight_matrix(10 ** 18))
print(f"brute n = 10: {bench(knight_brute, 10, 1) * 1e3:8.1f} ms (tăng ~2.2 lần mỗi n)")

n = 5000 dp            8.892 ms
n = 5000 symmetric     1.078 ms


n = 5000 matrix        1.715 ms


matrix n = 10^18:    9.108 ms -> 805313014


brute n = 10:      7.9 ms (tăng ~2.2 lần mỗi n)


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn `n = 1..11` với oracle liệt kê dãy theo hình học, (E) tính chất và kiểm chứng chéo, (F) giới hạn.

### A. Ví dụ của đề

In [16]:
def oracle(n):
    """Liệt kê MỌI dãy nhảy: nước đi tính trực tiếp từ hình học (±1,±2),(±2,±1) trên toạ độ bàn phím — không dùng danh sách kề viết tay."""
    def count(pos, left):
        if left == 0:
            return 1
        total = 0
        for dr, dc in KNIGHT:
            nxt = (pos[0] + dr, pos[1] + dc)
            if nxt in CELL:                                # chỉ đáp xuống ô số
                total += count(nxt, left - 1)
        return total

    return sum(count(POS[d], n - 1) for d in range(10)) % MOD


IMPLS = {
    "brute": knight_brute,
    "dp": knight_dp,
    "symmetric": knight_symmetric,
    "matrix": knight_matrix,
    "Solution": lambda n: Solution().knightDialer(n),
    "SolutionBrute": lambda n: SolutionBrute().knightDialer(n),
    "SolutionDP": lambda n: SolutionDP().knightDialer(n),
    "SolutionSymmetric": lambda n: SolutionSymmetric().knightDialer(n),
    "SolutionMatrix": lambda n: SolutionMatrix().knightDialer(n),
}


def check(n, want=None, names=None):
    if want is None:
        want = oracle(n)
    for name in (names or IMPLS):
        got = IMPLS[name](n)
        assert got == want, (name, n, got, want)


for n, want in examples[:2]:
    assert oracle(n) == want
    check(n, want)
for name in ("dp", "symmetric", "matrix", "Solution", "SolutionDP", "SolutionSymmetric", "SolutionMatrix"):
    assert IMPLS[name](3131) == 136006598, name
print("ví dụ đề: oracle hình học khớp n = 1, 2; 7 cài đặt nhanh cho n = 3131 đều là 136006598")

ví dụ đề: oracle hình học khớp n = 1, 2; 7 cài đặt nhanh cho n = 3131 đều là 136006598


### B. Edge case

- `n = 1`: `10`.
- `n = 2`: `20` (tổng bậc).
- `n = 3`: `46`; `n = 4`: `104`; `n = 5`: `240`.
- `n` lớn nhất `5000`.
- Kiểm tra từng chữ số xuất phát: số dãy bắt đầu từ `5` là `1` (n = 1) rồi `0`.
- Giá trị trước khi lấy modulo: `n = 40` đã vượt `10^9 + 7`.

In [17]:
known = {1: 10, 2: 20, 3: 46, 4: 104, 5: 240, 6: 544, 7: 1256, 8: 2848, 9: 6576, 10: 14912}
for n, want in known.items():
    check(n, want)
def exact(n):
    v = [1] * 10
    for _ in range(n - 1):
        v = [sum(v[s] * A[s][d] for s in range(10)) for d in range(10)]      # không modulo (số nguyên lớn của Python)
    return sum(v)
assert exact(40) > MOD and exact(40) % MOD == Solution().knightDialer(40)
assert Solution().knightDialer(5000) == knight_symmetric(5000) == knight_matrix(5000) == knight_dp(5000)
print(len(known), "giá trị đã biết (n = 1..10) đúng; n = 40 vượt MOD nhưng khớp sau modulo; n = 5000 khớp 4 cách")

10 giá trị đã biết (n = 1..10) đúng; n = 40 vượt MOD nhưng khớp sau modulo; n = 5000 khớp 4 cách


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [18]:
def dp_with(nxt, n, steps=None, mod=True, in_place=False):
    dp = [1] * 10
    for _ in range(n - 1 if steps is None else steps):
        if in_place:
            for s in range(10):
                for d in nxt[s]:
                    dp[d] = dp[d] + dp[s]                # SAI: ghi đè giá trị còn cần
        else:
            new = [0] * 10
            for s in range(10):
                for d in nxt[s]:
                    new[d] += dp[s]
            dp = new
        if mod:
            dp = [x % MOD for x in dp]
    return sum(dp) % MOD if mod else sum(dp)


def bug_wrong_adj(n):                                  # nhập sai danh sách kề: 4 thiếu 0
    nxt = dict(NXT)
    nxt[4] = [3, 9]
    return dp_with(nxt, n)


def bug_five_connected(n):                             # cho ô 5 nhảy được (sai)
    nxt = dict(NXT)
    nxt[5] = [1, 3]
    return dp_with(nxt, n)


def bug_off_by_one_steps(n):                           # n bước thay vì n - 1
    return dp_with(NXT, n, steps=n)


def bug_no_mod(n):                                     # quên modulo
    return dp_with(NXT, n, mod=False)


def bug_in_place(n):                                   # cập nhật tại chỗ
    return dp_with(NXT, n, in_place=True)


def bug_symmetric_wrong(n):                            # công thức đối xứng sai: c' = d + a (thiếu hệ số 2)
    if n == 1:
        return 10
    a = b = c = d = 1
    for _ in range(n - 1):
        a, b, c, d = (b + c) % MOD, 2 * a % MOD, (d + a) % MOD, 2 * c % MOD
    return (4 * a + 2 * b + 2 * c + d) % MOD


BUGS = [bug_wrong_adj, bug_five_connected, bug_off_by_one_steps, bug_no_mod, bug_in_place, bug_symmetric_wrong]
for bug in BUGS:
    bad = next((n for n in range(1, 60) if bug(n) != Solution().knightDialer(n)), None)
    assert bad is not None, bug.__name__
    print(f"{bug.__name__:22s} sai tại n = {bad}: ra {bug(bad)}, đúng {Solution().knightDialer(bad)}")

bug_wrong_adj          sai tại n = 2: ra 19, đúng 20
bug_five_connected     sai tại n = 2: ra 22, đúng 20
bug_off_by_one_steps   sai tại n = 1: ra 20, đúng 10
bug_no_mod             sai tại n = 24: ra 1609056256, đúng 609056249
bug_in_place           sai tại n = 2: ra 62, đúng 20
bug_symmetric_wrong    sai tại n = 2: ra 18, đúng 20


### D. Vét cạn `n = 1..11` với oracle hình học

Oracle liệt kê **mọi** dãy nhảy (số dãy tới `~7·10^4` ở `n = 11`) bằng đệ quy trên toạ độ.

In [19]:
t0 = time.perf_counter()
cnt = 0
for n in range(1, 12):
    want = oracle(n)
    for name in IMPLS:
        assert IMPLS[name](n) == want, (name, n)
    cnt += 1
print(cnt, "giá trị n: 9 cài đặt khớp oracle hình học (liệt kê dãy);", round(time.perf_counter() - t0, 1), "s")
# đếm riêng theo chữ số xuất phát bằng liệt kê, so với DP
def count_from(d, n):
    def go(pos, left):
        return 1 if left == 0 else sum(go((pos[0] + dr, pos[1] + dc), left - 1) for dr, dc in KNIGHT if (pos[0] + dr, pos[1] + dc) in CELL)
    return go(POS[d], n - 1)


for n in (1, 2, 5, 8):
    dp = [1] * 10
    for _ in range(n - 1):
        new = [0] * 10
        for s in range(10):
            for d in NXT[s]:
                new[d] += dp[s]
        dp = new
    # dp[d] = số dãy KẾT THÚC ở d; do đồ thị vô hướng, bằng số dãy BẮT ĐẦU ở d
    assert all(dp[d] == count_from(d, n) for d in range(10)), n
print("số dãy bắt đầu ở từng chữ số = số dãy kết thúc ở chữ số đó (đồ thị vô hướng): đúng với n = 1, 2, 5, 8")

11 giá trị n: 9 cài đặt khớp oracle hình học (liệt kê dãy); 0.1 s
số dãy bắt đầu ở từng chữ số = số dãy kết thúc ở chữ số đó (đồ thị vô hướng): đúng với n = 1, 2, 5, 8


### E. Tính chất và kiểm chứng chéo

(1) 4 cách nhanh khớp nhau với mọi `n ≤ 300`; (2) truy hồi tuyến tính: tổng `S(n)` thoả một truy hồi cấp `≤ 4` (kiểm chứng bằng đa thức đặc trưng của lớp đối xứng); (3) `S(n)` luôn **chẵn** với `n ≥ 2` (mỗi dãy có dãy đối xứng phản chiếu trái–phải khác nó, trừ khi chứa ô `5`... nên đếm cặp); (4) `S(m + n − 1) = Σ_{x, y} A^{m−1}[·, x] ... ` — tính kết hợp: tổng ma trận lũy thừa khớp tích.

In [20]:
for n in range(1, 301):
    r = {k: IMPLS[k](n) for k in ("dp", "symmetric", "matrix", "Solution", "SolutionSymmetric", "SolutionMatrix")}
    assert len(set(r.values())) == 1, (n, r)
print("(1) 6 cài đặt khớp nhau với mọi n = 1..300")

# (2) truy hồi: với S(n) chính xác (không modulo) tìm hệ số bằng đại số: dùng ma trận đặc trưng của 4 biến đối xứng
def exact_sym(n):
    if n == 1:
        return 10
    a = b = c = d = 1
    for _ in range(n - 1):
        a, b, c, d = b + c, 2 * a, d + 2 * a, 2 * c
    return 4 * a + 2 * b + 2 * c + d


# thử truy hồi cấp 4 nhỏ nhất bằng giải hệ trên n = 2..: tìm hằng số nguyên p1..p4 sao cho S(n) = p1 S(n-1) + ... + p4 S(n-4)
import fractions
rows = [[fractions.Fraction(exact_sym(n - i)) for i in range(1, 5)] + [fractions.Fraction(exact_sym(n))] for n in range(6, 10)]
for col in range(4):                                      # khử Gauss
    piv = next(r for r in range(col, 4) if rows[r][col] != 0)
    rows[col], rows[piv] = rows[piv], rows[col]
    rows[col] = [x / rows[col][col] for x in rows[col]]
    for r in range(4):
        if r != col:
            rows[r] = [x - rows[r][col] * y for x, y in zip(rows[r], rows[col])]
coef = [rows[i][4] for i in range(4)]
print("(2) hệ số truy hồi S(n) = p1 S(n-1) + ... + p4 S(n-4):", [str(x) for x in coef])
assert all(sum(coef[i - 1] * exact_sym(n - i) for i in range(1, 5)) == exact_sym(n) for n in range(6, 40))
for n in range(2, 200):
    assert exact_sym(n) % 2 == 0                          # (3) chẵn với n >= 2
print("(3) S(n) chẵn với mọi n = 2..199; truy hồi cấp 4 đúng cho n = 6..39")
M = lambda e: mat_pow(A, e)
for m, n in [(3, 5), (10, 20), (100, 3)]:
    left = sum(map(sum, mat_pow(A, m - 1 + n - 1)))
    right = sum(sum(mat_mul(mat_pow(A, m - 1), mat_pow(A, n - 1))[i]) for i in range(10))
    assert left % MOD == right % MOD
print("(4) A^(m-1) · A^(n-1) = A^(m+n-2): đúng")

(1) 6 cài đặt khớp nhau với mọi n = 1..300
(2) hệ số truy hồi S(n) = p1 S(n-1) + ... + p4 S(n-4): ['0', '6', '0', '-4']
(3) S(n) chẵn với mọi n = 2..199; truy hồi cấp 4 đúng cho n = 6..39
(4) A^(m-1) · A^(n-1) = A^(m+n-2): đúng


### F. Giới hạn: `n = 5000`

Bốn cách nhanh khớp nhau với `n = 5000`; kiểm tra `n = 1` biên và ví dụ `3131`; luỹ thừa ma trận với `n` cực lớn khớp tính chất tuần hoàn của dãy modulo (khớp DP với `n` ngẫu nhiên trong `[1, 5000]`).

In [21]:
rng = random.Random(500)
fast = ["dp", "symmetric", "matrix", "Solution", "SolutionDP", "SolutionSymmetric", "SolutionMatrix"]
for _ in range(60):
    n = rng.randint(1, 5000)
    res = {k: IMPLS[k](n) for k in fast}
    assert len(set(res.values())) == 1, (n, res)
assert all(IMPLS[k](5000) == knight_dp(5000) for k in fast)
assert all(IMPLS[k](1) == 10 for k in fast)
print("60 giá trị n ngẫu nhiên trong [1, 5000]: 7 cài đặt nhanh khớp nhau; n = 5000 =", Solution().knightDialer(5000))

60 giá trị n ngẫu nhiên trong [1, 5000]: 7 cài đặt nhanh khớp nhau; n = 5000 = 406880451


## 13. Các lỗi người mới hay gặp

### Lỗi 1: nhập sai danh sách kề (thiếu một nước đi)

In [22]:
print("n = 2:", bug_wrong_adj(2), "| đúng:", Solution().knightDialer(2))
assert bug_wrong_adj(2) == 19

n = 2: 19 | đúng: 20


**Vì sao sai:** thiếu cạnh `4 → 0` bớt một số. **Đúng:** tính danh sách kề **từ hình học** (`moves_from_geometry`) hoặc kiểm tra tổng bậc `= 20`.

### Lỗi 2: cho ô `5` có nước đi

In [23]:
print("n = 2:", bug_five_connected(2), "| đúng:", Solution().knightDialer(2))
assert bug_five_connected(2) == 22

n = 2: 22 | đúng: 20


**Vì sao sai:** ô `5` ở giữa nên quân mã không nhảy được tới ô nào (và không ô nào nhảy tới `5`). **Đúng:** `nxt[5] = []`.

### Lỗi 3: số bước `n` thay vì `n − 1`

In [24]:
print("n = 1:", bug_off_by_one_steps(1), "| đúng:", Solution().knightDialer(1))
assert bug_off_by_one_steps(1) == 20

n = 1: 20 | đúng: 10


**Vì sao sai:** số có `n` chữ số cần `n − 1` bước nhảy. **Đúng:** `range(n - 1)`.

### Lỗi 4: quên modulo

In [25]:
print("n = 3131: số chữ số của kết quả không mod =", len(str(bug_no_mod(3131))), "| đúng:", Solution().knightDialer(3131))
assert bug_no_mod(3131) != Solution().knightDialer(3131)

n = 3131: số chữ số của kết quả không mod = 1127 | đúng: 136006598


**Vì sao sai:** đáp án tăng theo cấp số nhân (`~2.2^n`); ở ngôn ngữ 32/64-bit sẽ tràn. **Đúng:** `% MOD` sau mỗi phép cộng.

### Lỗi 5: cập nhật mảng DP tại chỗ

In [26]:
print("n = 3:", bug_in_place(3), "| đúng:", Solution().knightDialer(3))
assert bug_in_place(3) != Solution().knightDialer(3)

n = 3: 444 | đúng: 46


**Vì sao sai:** `dp[s]` còn cần cho nhiều ô đích; ghi đè làm các ô sau đọc giá trị đã bị đổi. **Đúng:** dùng mảng mới `new`.

### Lỗi 6 (bonus): công thức đối xứng sai hệ số

In [27]:
print("n = 4:", bug_symmetric_wrong(4), "| đúng:", Solution().knightDialer(4))
assert bug_symmetric_wrong(4) != Solution().knightDialer(4)

n = 4: 72 | đúng: 104


**Vì sao sai:** ô `4` nhận từ `0` **và hai** ô góc (`3`, `9`) nên `c' = d + 2a`. **Đúng:** đối chiếu với DP đầy đủ trước khi dùng công thức rút gọn.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "đếm dãy bước trên đồ thị cố định có số đỉnh nhỏ, độ dài `n` lớn" → **DP theo đỉnh cuối** và **luỹ thừa ma trận kề** (truy hồi tuyến tính).

**Mẹo:**

1. Dựng danh sách kề **bằng hình học** rồi in ra để kiểm tra (tổng bậc `= 20`).
2. Số bước `= n − 1`.
3. Dùng mảng DP **mới** mỗi bước.
4. Nhận ra đối xứng để rút gọn trạng thái (nhưng luôn đối chiếu với bản đầy đủ).
5. `n` lên `10^18` ⇒ luỹ thừa ma trận `O(k³ log n)`.
6. Modulo sau mỗi phép cộng/nhân.

**Họ bài cùng pattern:** Number of Ways to Stay in the Same Place After Some Steps (1269), Count Vowels Permutation (1220), Knight Probability in Chessboard (688), Out of Boundary Paths (576), Fibonacci / Climbing Stairs (70), Domino and Tromino Tiling (790).

## 15. Biến thể

1. **Quân khác:** xe, tượng, vua trên cùng bàn phím (đổi quy tắc nước đi).
2. **Bắt đầu ở chữ số cố định `s`** (hoặc kết thúc ở `e`).
3. **Số số phân biệt có độ dài `≤ n`** (tổng cộng dồn).
4. **Cấm một số ô** (hỏng phím).
5. **Đếm số dãy không chứa chữ số `x`.**

Cài biến thể 2, 3, 4 (đối chiếu liệt kê hình học):

In [28]:
def count_with(n, start=None, end=None, banned=frozenset(), total_up_to=False):
    def build_adj():
        return {d: [t for t in moves_from_geometry(d) if t not in banned] for d in range(10) if d not in banned}

    adj = build_adj()

    def run(length):
        dp = {d: (1 if start is None or d == start else 0) for d in adj}
        for _ in range(length - 1):
            new = {d: 0 for d in adj}
            for s in adj:
                for t in adj[s]:
                    new[t] += dp[s]
            dp = new
        return sum(v for d, v in dp.items() if end is None or d == end)

    return sum(run(k) for k in range(1, n + 1)) if total_up_to else run(n)


def brute_with(n, start=None, end=None, banned=frozenset()):
    def go(d, left):
        if left == 0:
            return 1 if end is None or d == end else 0
        return sum(go(t, left - 1) for t in moves_from_geometry(d) if t not in banned)

    starts = [d for d in range(10) if d not in banned and (start is None or d == start)]
    return sum(go(d, n - 1) for d in starts)


for n in range(1, 8):
    assert count_with(n) == Solution().knightDialer(n) == brute_with(n)
    for s in range(10):
        assert count_with(n, start=s) == brute_with(n, start=s)
        assert count_with(n, start=s, end=(s + 3) % 10) == brute_with(n, start=s, end=(s + 3) % 10)
    ban = frozenset({4, 6})
    assert count_with(n, banned=ban) == brute_with(n, banned=ban)
    assert count_with(n, total_up_to=True) == sum(Solution().knightDialer(k) for k in range(1, n + 1))
print("biến thể: bắt đầu/kết thúc cố định, cấm phím, tổng độ dài <= n: khớp liệt kê hình học (n = 1..7)")
assert count_with(2, start=4) == 3 and count_with(3, banned=frozenset({5})) == Solution().knightDialer(3) - 0

biến thể: bắt đầu/kết thúc cố định, cấm phím, tổng độ dài <= n: khớp liệt kê hình học (n = 1..7)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Liệt kê dãy | DFS mọi đường đi | `O(2.x^n)` | `O(n)` | oracle |
| 2. DP chữ số cuối | `dp_k[d] = Σ dp_{k−1}[s]` | `O(20 n)` | `O(1)` | **đáp án chính** |
| 3. DP đối xứng | 4 trạng thái | `O(n)` | `O(1)` | nhanh, cần chứng minh |
| 4. Luỹ thừa ma trận | `A^{n−1}` | `O(10³ log n)` | `O(100)` | `n` rất lớn |

Cả bốn khớp oracle hình học trên `n = 1..11` và khớp nhau tới `n = 5000` (mục 12).

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [29]:
print(Solution().knightDialer(3), Solution().knightDialer(4), count_with(3, start=0), count_with(3, start=5))

46 104 6 0


### Bài 2 (tính tay)

Tính `dp` cho `n = 3`: (a) `dp_2` (kết thúc ở `0..9`); (b) `dp_3`; (c) tổng. Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện DP theo chữ số cuối:

In [30]:
def dp_todo(n):
    nxt = [[4, 6], [6, 8], [7, 9], [4, 8], [0, 3, 9], [], [0, 1, 7], [2, 6], [1, 3], [2, 4]]
    dp = [1] * 10
    for _ in range(n - 1):
        new = [0] * 10
        for s in range(10):
            pass        # TODO: với mỗi d trong nxt[s]: new[d] = (new[d] + dp[s]) % MOD
        dp = new
    return 0            # TODO: sum(dp) % MOD


# Khi làm xong, bỏ comment để kiểm tra:
# assert [dp_todo(n) for n, _ in examples] == [10, 20, 136006598]

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **luỹ thừa ma trận**:

In [31]:
class MySolution:
    def knightDialer(self, n):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().knightDialer(n) == w for n, w in examples)

### Bài 5 (mini challenge): Count Vowels Permutation (1220)

Đếm số chuỗi độ dài `n` gồm các nguyên âm `a, e, i, o, u` với quy tắc: sau `a` chỉ là `e`; sau `e` là `a` hoặc `i`; sau `i` là mọi nguyên âm **trừ** `i`; sau `o` là `i` hoặc `u`; sau `u` chỉ là `a`. Trả về modulo `10^9 + 7`. Viết `count_vowel_permutation(n)` bằng DP và kiểm tra bằng liệt kê.

In [32]:
def count_vowel_permutation(n):
    pass            # TODO


# Khi làm xong:
# assert count_vowel_permutation(1) == 5 and count_vowel_permutation(2) == 10 and count_vowel_permutation(5) == 68

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: DP theo chữ số cuối.
- Mức 2: `n = 3` ⇒ `46`; `n = 4` ⇒ `104`.
- Mức 3: kết quả `46 104 6 0`: các số 3 chữ số bắt đầu bằng `0` có `6`; bắt đầu bằng `5` không có (`n ≥ 2`).

**Bài 2**

- Mức 1: `dp_2 = [2,2,2,2,3,0,3,2,2,2]`.
- Mức 2: `dp_3[0] = dp_2[4] + dp_2[6] = 6`, `dp_3[1] = dp_2[6] + dp_2[8] = 5`, `dp_3[2] = dp_2[7] + dp_2[9] = 4`.
- Mức 3: `dp_3 = [6,5,4,5,6,0,6,5,4,5]`, tổng `46`.

**Bài 3**

- Mức 1: `for d in nxt[s]: new[d] += dp[s]`.
- Mức 2: lấy modulo.
- Mức 3: `return sum(dp) % MOD`.

**Bài 4**

- Mức 1: ma trận kề `10 × 10`.
- Mức 2: luỹ thừa nhanh `A^(n-1)`.
- Mức 3: tổng mọi phần tử.

**Bài 5**

- Mức 1: `dp[v]` = số chuỗi kết thúc bằng nguyên âm `v`.
- Mức 2: `a' = e + i + u`, `e' = a + i`, `i' = e + o`, `o' = i`, `u' = i + o`.
- Mức 3: tổng 5 giá trị, modulo.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [33]:
vals = (Solution().knightDialer(3), Solution().knightDialer(4), count_with(3, start=0), count_with(3, start=5))
assert vals == (46, 104, 6, 0) == (oracle(3), oracle(4), brute_with(3, start=0), brute_with(3, start=5))
print("Bài 1 OK:", vals)

Bài 1 OK: (46, 104, 6, 0)


### Bài 2

`dp_2 = [2,2,2,2,3,0,3,2,2,2]` (tổng 20); `dp_3 = [6,5,4,5,6,0,6,5,4,5]`; tổng `46`.

In [34]:
dp = [1] * 10
seq = []
for _ in range(2):
    new = [0] * 10
    for s in range(10):
        for d in NXT[s]:
            new[d] += dp[s]
    dp = new
    seq.append(dp)
assert seq[0] == [2, 2, 2, 2, 3, 0, 3, 2, 2, 2] and seq[1] == [6, 5, 4, 5, 6, 0, 6, 5, 4, 5] and sum(seq[1]) == 46
print("Bài 2 OK: dp_2 =", seq[0], "| dp_3 =", seq[1], "| tổng =", sum(seq[1]))

Bài 2 OK: dp_2 = [2, 2, 2, 2, 3, 0, 3, 2, 2, 2] | dp_3 = [6, 5, 4, 5, 6, 0, 6, 5, 4, 5] | tổng = 46


### Bài 3

In [35]:
def dp_answer(n):
    nxt = [[4, 6], [6, 8], [7, 9], [4, 8], [0, 3, 9], [], [0, 1, 7], [2, 6], [1, 3], [2, 4]]
    dp = [1] * 10
    for _ in range(n - 1):
        new = [0] * 10
        for s in range(10):
            for d in nxt[s]:
                new[d] = (new[d] + dp[s]) % MOD
        dp = new
    return sum(dp) % MOD


assert [dp_answer(n) for n, _ in examples] == [10, 20, 136006598]
for n in range(1, 12):
    assert dp_answer(n) == oracle(n)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [36]:
class MySolutionAnswer:
    def knightDialer(self, n):
        nxt = [[4, 6], [6, 8], [7, 9], [4, 8], [0, 3, 9], [], [0, 1, 7], [2, 6], [1, 3], [2, 4]]
        M = [[0] * 10 for _ in range(10)]
        for s in range(10):
            for d in nxt[s]:
                M[s][d] = 1

        def mul(X, Y):
            return [[sum(X[i][k] * Y[k][j] for k in range(10)) % MOD for j in range(10)] for i in range(10)]

        R = [[int(i == j) for j in range(10)] for i in range(10)]
        e = n - 1
        while e:
            if e & 1:
                R = mul(R, M)
            M = mul(M, M)
            e >>= 1
        return sum(map(sum, R)) % MOD


assert all(MySolutionAnswer().knightDialer(n) == w for n, w in examples)
for n in range(1, 12):
    assert MySolutionAnswer().knightDialer(n) == oracle(n)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [37]:
def count_vowel_permutation_answer(n):
    a = e = i = o = u = 1
    for _ in range(n - 1):
        a, e, i, o, u = (e + i + u) % MOD, (a + i) % MOD, (e + o) % MOD, i, (i + o) % MOD
    return (a + e + i + o + u) % MOD


def count_vowel_permutation_brute(n):
    rules = {"a": "e", "e": "ai", "i": "aeou", "o": "iu", "u": "a"}
    def go(c, left):
        return 1 if left == 0 else sum(go(nc, left - 1) for nc in rules[c])
    return sum(go(c, n - 1) for c in "aeiou") % MOD


assert [count_vowel_permutation_answer(n) for n in (1, 2, 5)] == [5, 10, 68]
for n in range(1, 11):
    assert count_vowel_permutation_answer(n) == count_vowel_permutation_brute(n)
print("Bài 5 OK: khớp liệt kê")

Bài 5 OK: khớp liệt kê


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (đếm đường đi độ dài `n` trên đồ thị nhỏ → DP theo đỉnh cuối):**

```python
class Solution:
    def knightDialer(self, n: int) -> int:
        MOD = 10 ** 9 + 7
        nxt = [[4, 6], [6, 8], [7, 9], [4, 8], [0, 3, 9], [], [0, 1, 7], [2, 6], [1, 3], [2, 4]]
        dp = [1] * 10
        for _ in range(n - 1):
            new = [0] * 10
            for s in range(10):
                for d in nxt[s]:
                    new[d] = (new[d] + dp[s]) % MOD
            dp = new
        return sum(dp) % MOD
```

**Checklist trước khi submit:**

- [ ] Danh sách kề đúng (tổng bậc `20`, ô `5` rỗng)?
- [ ] Số bước `n − 1` (n = 1 ⇒ `10`)?
- [ ] Mảng DP **mới** mỗi bước (không cập nhật tại chỗ)?
- [ ] `% (10**9 + 7)` sau mỗi phép cộng?
- [ ] Đã thử `n = 1, 2, 3, 4`, `n = 3131 → 136006598`, `n = 5000`?
- [ ] Đúng tên `class Solution`, `knightDialer(self, n)`, không in debug.